# B. Validate, tune and write `submission.csv` (U-Net detector + flow-aware linking)

Inputs: the competition data, a dataset containing `unet_detector.pt` (from notebook A), and your `zarr-wheels` dataset. Internet **off**.

1. **Validation (Phase 0)** on ~20 held-out training movies, using 4-frame windows spread over the whole movie (early *and* late, sparse *and* dense), reporting TP/FP/FN, node recall and edge Jaccard, and comparing the old blob detector with the U-Net, with and without motion compensation.
2. **Tuning:** picks the detection threshold, link distance and flow on/off with the best pooled edge Jaccard on those windows (ties go to fewer nodes).
3. **Linking (Phase 2):** each cell's expected position is corrected by the local motion field (median displacement of its matched neighbours), then Hungarian matching, safe divisions and 1-frame gap closing.
4. **Submission:** every test movie -> nodes and edges -> `submission.csv`. If the weights are missing or anything fails, it falls back to the blob detector and still writes a file.

## 1. Setup (paths, offline-safe zarr)

In [1]:
import os, sys, time, math, random, subprocess, traceback, fnmatch, importlib, json, copy
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import version as _pkg_version, PackageNotFoundError
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

COMPETITION = "biohub-cell-tracking-during-development"
_override = os.environ.get("BIOHUB_COMP_DIR")
COMP_DIR = Path(_override) if _override else next(
    (p for p in [Path(f"/kaggle/input/competitions/{COMPETITION}"), Path(f"/kaggle/input/{COMPETITION}")] if p.exists()),
    Path(f"/kaggle/input/competitions/{COMPETITION}"))
TRAIN_DIR, TEST_DIR = COMP_DIR / "train", COMP_DIR / "test"
SAMPLE_SUBMISSION = COMP_DIR / "sample_submission.csv"
_w = os.environ.get("BIOHUB_WORK_DIR")
WORKING_DIR = Path(_w) if _w else (Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("."))
print("COMP_DIR:", COMP_DIR, "| exists:", COMP_DIR.exists(), "| train:", TRAIN_DIR.exists(), "| test:", TEST_DIR.exists())


def find_files(root, patterns, max_depth=6, skip_names=()):
    """Walk `root` (never into movie folders) and return files matching any of `patterns`."""
    out, root = [], str(root)
    if not os.path.isdir(root):
        return out
    base = root.rstrip(os.sep).count(os.sep)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath.count(os.sep) - base
        dirnames[:] = [d for d in dirnames if d not in skip_names and not d.endswith((".zarr", ".geff")) and depth < max_depth]
        out += [os.path.join(dirpath, f) for f in filenames if any(fnmatch.fnmatch(f, p) for p in patterns)]
    return out


def ensure_zarr():
    """Scoring runs with Internet OFF. Need zarr >= 3: use it if installed, else install from an attached wheels dataset."""
    def ok():
        try:
            return int(_pkg_version("zarr").split(".")[0]) >= 3
        except PackageNotFoundError:
            return False
    try:
        print("installed zarr:", _pkg_version("zarr"), "(need >= 3)")
    except PackageNotFoundError:
        print("installed zarr: none (need >= 3)")
    if ok():
        return
    wheel_dirs = sorted({os.path.dirname(w) for w in find_files("/kaggle/input", ["zarr-*.whl"], skip_names=(COMPETITION,))})
    print("folders with zarr wheels under /kaggle/input:", wheel_dirs or "none -> attach the zarr-wheels dataset (Add Input)")
    for d in wheel_dirs:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", d, "zarr>=3.0.10,<4"],
                           capture_output=True, text=True)
        importlib.invalidate_caches()
        print("offline zarr install from", d, "->", "ok" if r.returncode == 0 else r.stderr[-200:])
        if ok():
            return
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--retries", "1", "--timeout", "10", "zarr>=3.0.10,<4"],
                   capture_output=True, text=True)          # only works with Internet ON
    importlib.invalidate_caches()
    if not ok():
        raise RuntimeError("zarr >= 3 is not available. Attach your zarr-wheels dataset (Add Input) and run with Internet OFF, "
                           "or turn Internet ON for an interactive/training run.")

ensure_zarr()
import zarr
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| zarr", zarr.__version__, "| device:", device)


COMP_DIR: /kaggle/input/competitions/biohub-cell-tracking-during-development | exists: True | train: True | test: True
installed zarr: none (need >= 3)
folders with zarr wheels under /kaggle/input: ['/kaggle/input/notebooks/parth2157/zarr-wheels/zarr_wheels']
offline zarr install from /kaggle/input/notebooks/parth2157/zarr-wheels/zarr_wheels -> ok
torch 2.11.0+cpu | zarr 3.4.2 | device: cpu


## 2. Settings and helpers

In [2]:
# ============================ SETTINGS ============================
DETECTOR = "unet"          # "unet" (needs unet_detector.pt) or "blob" (no weights needed)
THRESHOLD = 0.2            # heat-map threshold; AUTO_TUNE overwrites it
MAX_LINK_UM = 10.0         # maximum distance between a cell's expected position and its match; AUTO_TUNE overwrites it
USE_FLOW = True            # motion-compensated linking; AUTO_TUNE overwrites it
K_MAD = 4.0                # blob-detector sensitivity (only for DETECTOR="blob" and for the comparison)
AUTO_TUNE = True
N_HOLDOUT = 20             # must equal the value used in notebook A
SEED = 0                   # must equal the value used in notebook A
WINDOW_STARTS = (0, 24, 48, 72, 95)   # first frame of each validation window
WINDOW_WIDTH = 4
THR_GRID = (0.1, 0.15, 0.2, 0.3, 0.4)
LINK_GRID = (8.0, 10.0, 12.0)
COMPARE_BLOB = True        # also score the old blob detector on the same windows (first 6 held-out movies)
# ==================================================================


In [3]:
def list_stems(d):
    return sorted(p.name[:-5] for p in Path(d).glob("*.zarr")) if Path(d).exists() else []


def open_movie(split_dir, stem):
    """Returns (zarr array (T,Z,Y,X), voxel_scale (z,y,x) in micrometres)."""
    store = zarr.open(str(Path(split_dir) / f"{stem}.zarr"), mode="r")
    scale = store.attrs["multiscales"][0]["datasets"][0]["coordinateTransformations"][0]["scale"]
    return store["0"], tuple(scale[1:])


def load_geff_graph(geff_path):
    """Load a .geff annotation graph into (nodes_df, edges_df)."""
    store = zarr.open(str(geff_path), mode="r")
    nodes_df = pd.DataFrame({"node_id": store["nodes/ids"][:], "t": store["nodes/props/t/values"][:],
                             "z": store["nodes/props/z/values"][:], "y": store["nodes/props/y/values"][:],
                             "x": store["nodes/props/x/values"][:]})
    ids = store["edges/ids"][:]
    return nodes_df, pd.DataFrame({"source_id": ids[:, 0], "target_id": ids[:, 1]})


def split_stems(stems, n_hold=20, seed=0):
    """Deterministic hold-out split. Identical code in the training and inference notebooks, so both agree on it."""
    stems = sorted(stems)
    n = min(n_hold, max(len(stems) // 5, 1)) if len(stems) > 1 else 0
    hold = set(random.Random(seed).sample(stems, n))
    return [s for s in stems if s not in hold], sorted(hold)


def norm_frame(frame):
    """Robust per-frame intensity normalisation. The same function is used for training and for inference."""
    f = np.asarray(frame, dtype=np.float32)
    lo, hi = np.percentile(f[::2, ::4, ::4], [1.0, 99.8])
    return ((f - np.float32(lo)) / np.float32(max(float(hi - lo), 1e-6))).astype(np.float32)


## 3. U-Net model and decoding (must match notebook A)

In [4]:
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(cin, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.LeakyReLU(0.1, inplace=True),
            nn.Conv3d(cout, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.LeakyReLU(0.1, inplace=True))
    def forward(self, x):
        return self.net(x)


class UNet3D(nn.Module):
    """Cell-centre heat-map network.  Input (B,1,Z,Y,X)  ->  logits (B,1,Z,Y/2,X/2).
    Down-sampling is anisotropic because z voxels (1.6 um) are 4x coarser than y/x voxels (0.4 um):
    stem (1,2,2), then (1,2,2), (2,2,2), (2,2,2), (2,2,2).  Z must be a multiple of 8, Y and X multiples of 32
    (use pad_to_multiple). All channel counts must be multiples of 8 (GroupNorm)."""
    STRIDES = [(1, 2, 2), (1, 2, 2), (2, 2, 2), (2, 2, 2), (2, 2, 2)]

    def __init__(self, ch=(24, 48, 96, 144, 192)):
        super().__init__()
        self.ch = tuple(ch)
        assert len(self.ch) == 5 and all(c % 8 == 0 for c in self.ch)
        self.stem = nn.Sequential(nn.Conv3d(1, ch[0], 3, stride=(1, 2, 2), padding=1, bias=False),
                                  nn.GroupNorm(8, ch[0]), nn.LeakyReLU(0.1, inplace=True))
        self.enc0 = ConvBlock(ch[0], ch[0])
        self.downs, self.encs = nn.ModuleList(), nn.ModuleList()
        for i in range(1, 5):
            self.downs.append(nn.Sequential(nn.Conv3d(ch[i - 1], ch[i], 3, stride=self.STRIDES[i], padding=1, bias=False),
                                            nn.GroupNorm(8, ch[i]), nn.LeakyReLU(0.1, inplace=True)))
            self.encs.append(ConvBlock(ch[i], ch[i]))
        self.decs = nn.ModuleList([ConvBlock(ch[i] + ch[i - 1], ch[i - 1]) for i in range(4, 0, -1)])
        self.head = nn.Conv3d(ch[0], 1, 1)

    def forward(self, x):
        h = self.enc0(self.stem(x))
        skips = [h]
        for down, enc in zip(self.downs, self.encs):
            h = enc(down(h))
            skips.append(h)
        for k, dec in enumerate(self.decs):
            lvl = 4 - k                                                   # level of the current h
            h = F.interpolate(h, size=skips[lvl - 1].shape[2:], mode="trilinear", align_corners=False)
            h = dec(torch.cat([h, skips[lvl - 1]], dim=1))
        return self.head(h)


def pad_to_multiple(x, mz=8, my=32, mx=32):
    Z, Y, X = x.shape[-3:]
    pz, py, px = (-Z) % mz, (-Y) % my, (-X) % mx
    if pz or py or px:
        x = F.pad(x, (0, px, 0, py, 0, pz))
    return x, (Z, Y, X)


@torch.no_grad()
def detect_frame(model, frame, scale, thr=0.2, min_dist_um=3.0, max_peaks=8000):
    """One 3D frame -> (points (N,3) full-resolution voxel z,y,x ; scores (N,)). Peaks are found on the logits
    (no saturation ties), then thresholded by probability."""
    x = torch.from_numpy(norm_frame(frame))[None, None].to(device)
    x, (Z, Y, X) = pad_to_multiple(x)
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
        logits = model(x)
    logits = logits.float()[..., :Z, :(Y + 1) // 2, :(X + 1) // 2]
    size = (2 * max(int(round(min_dist_um / scale[0])), 1) + 1,
            2 * max(int(round(min_dist_um / (2 * scale[1]))), 1) + 1,
            2 * max(int(round(min_dist_um / (2 * scale[2]))), 1) + 1)
    mx = F.max_pool3d(logits, size, stride=1, padding=tuple(k // 2 for k in size))
    prob = torch.sigmoid(logits)
    idx = torch.nonzero((logits == mx) & (prob >= thr))[:, 2:]
    sc = prob[0, 0][idx[:, 0], idx[:, 1], idx[:, 2]]
    if len(idx) > max_peaks:
        top = torch.topk(sc, max_peaks).indices
        idx, sc = idx[top], sc[top]
    return (idx.cpu().numpy() * np.array([1, 2, 2])).astype(np.int64), sc.cpu().numpy()


def load_detector(path):
    ckpt = torch.load(path, map_location=device)
    model = UNet3D(ch=ckpt["config"]["ch"]).to(device)
    model.load_state_dict(ckpt["state_dict"])
    return model.eval()


## 4. Blob detector (fallback and baseline for comparison)

In [5]:
from concurrent.futures import ThreadPoolExecutor
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components


def classical_peaks(vol, scale, sigma_small_um=2.0, sigma_large_um=4.5, min_dist_um=3.0, k_mad=4.0, max_peaks=6000):
    """Blob detector for one 3D frame: difference of Gaussians, then local maxima above a noise-based threshold.
    Returns an (N,3) integer array of (z,y,x) voxel positions. No training needed."""
    v = np.asarray(vol, dtype=np.float32)
    lo, hi = np.percentile(v, [1.0, 99.8])
    v = (v - lo) / max(float(hi - lo), 1e-6)
    s_small = [sigma_small_um / s for s in scale]
    s_large = [sigma_large_um / s for s in scale]
    dog = gaussian_filter(v, s_small, truncate=3.0) - gaussian_filter(v, s_large, truncate=3.0)
    med = float(np.median(dog))
    mad = float(np.median(np.abs(dog - med))) * 1.4826 + 1e-6
    size = tuple(2 * max(int(round(min_dist_um / s)), 1) + 1 for s in scale)
    is_peak = (dog == maximum_filter(dog, size=size)) & (dog > med + k_mad * mad)
    pk = np.argwhere(is_peak)
    if len(pk) > max_peaks:
        pk = pk[np.argsort(-dog[tuple(pk.T)])[:max_peaks]]
    return pk


def points_to_nodes(points_by_t):
    """{t: (N,3) voxel array} -> nodes_df with unique node_id (starting at 1)."""
    rows, nid = [], 1
    for t in sorted(points_by_t):
        for z, y, x in points_by_t[t]:
            rows.append((nid, int(t), int(z), int(y), int(x)))
            nid += 1
    return pd.DataFrame(rows, columns=["node_id", "t", "z", "y", "x"], dtype="int64")


GPU_BLOB, GPU_CHECKED = False, False


def classical_peaks_gpu(vol, scale, sigma_small_um=2.0, sigma_large_um=4.5, min_dist_um=3.0, k_mad=4.0, max_peaks=6000):
    """Same blob detector as classical_peaks, but the Gaussian blurs and the local-maximum search run on the GPU."""
    import torch.nn.functional as F
    v = torch.as_tensor(np.asarray(vol), device=device).float()
    q = torch.quantile(v.flatten(), torch.tensor([0.01, 0.998], device=device))
    v = ((v - q[0]) / torch.clamp(q[1] - q[0], min=1e-6))[None, None]          # (1,1,Z,Y,X)

    def blur(x, sig_vox):
        for ax, sg in enumerate(sig_vox):                                       # ax: 0=z, 1=y, 2=x
            r = max(int(3.0 * sg + 0.5), 1)
            xs = torch.arange(-r, r + 1, device=device, dtype=torch.float32)
            k = torch.exp(-0.5 * (xs / sg) ** 2)
            k = k / k.sum()
            shape = [1, 1, 1, 1, 1]
            shape[2 + ax] = -1
            pad = [0] * 6
            pad[2 * (2 - ax)] = pad[2 * (2 - ax) + 1] = r
            x = F.conv3d(F.pad(x, pad, mode="reflect"), k.view(shape))
        return x

    dog = blur(v, [sigma_small_um / s for s in scale]) - blur(v, [sigma_large_um / s for s in scale])
    flat = dog.flatten()
    med = flat.median()
    mad = (flat - med).abs().median() * 1.4826 + 1e-6
    size = tuple(2 * max(int(round(min_dist_um / s)), 1) + 1 for s in scale)
    mx = F.max_pool3d(dog, kernel_size=size, stride=1, padding=tuple(k // 2 for k in size))
    pk = torch.nonzero((dog == mx) & (dog > med + k_mad * mad))[:, 2:]
    if len(pk) > max_peaks:
        strength = dog[0, 0][pk[:, 0], pk[:, 1], pk[:, 2]]
        pk = pk[torch.topk(strength, max_peaks).indices]
    return pk.cpu().numpy()


def check_gpu_blob(movie, scale):
    """Use the GPU blob detector only if it agrees with the CPU one on a real frame (and is actually faster)."""
    global GPU_BLOB, GPU_CHECKED
    GPU_CHECKED, GPU_BLOB = True, False
    if device.type != "cuda":
        return
    try:
        frame = np.asarray(movie[0])
        vs = np.asarray(scale, dtype=float)
        t0 = time.time(); a = classical_peaks(frame, scale, k_mad=K_MAD); t_cpu = time.time() - t0
        classical_peaks_gpu(frame, scale, k_mad=K_MAD); torch.cuda.synchronize()          # warm-up
        t0 = time.time(); b = classical_peaks_gpu(frame, scale, k_mad=K_MAD); torch.cuda.synchronize(); t_gpu = time.time() - t0
        d, _ = cKDTree(b * vs).query(a * vs) if len(a) and len(b) else (np.array([np.inf]), None)
        agree, ratio = float((d <= 2.0).mean()), len(b) / max(len(a), 1)
        GPU_BLOB = agree >= 0.9 and 0.8 <= ratio <= 1.25 and t_gpu < t_cpu
        print(f"GPU blob check: {len(a)} CPU peaks vs {len(b)} GPU peaks, agreement {agree:.2f}, "
              f"{t_cpu:.2f}s CPU vs {t_gpu:.2f}s GPU -> {'using GPU' if GPU_BLOB else 'using CPU'}")
    except Exception:
        traceback.print_exc()
        GPU_BLOB = False
        print("GPU blob check failed -> using CPU")


def classical_detect_movie(movie, scale, k_mad=None, verbose=True):
    """Blob detection for every frame. Frames are read by background threads while the GPU (or CPU) processes."""
    k_mad = K_MAD if k_mad is None else k_mad
    if not GPU_CHECKED:
        check_gpu_blob(movie, scale)
    fn = classical_peaks_gpu if GPU_BLOB else classical_peaks
    n_frames, pts = movie.shape[0], {}
    try:
        with ThreadPoolExecutor(max_workers=3) as pool:
            for t, frame in enumerate(pool.map(lambda i: np.asarray(movie[i]), range(n_frames))):
                pts[t] = fn(frame, scale, k_mad=k_mad)
                if verbose and (t + 1) % 25 == 0:
                    print(f"   blob detector: frame {t + 1}/{n_frames}")
    except Exception:
        traceback.print_exc()
        print("   threaded/GPU detection failed -> redoing this movie sequentially on the CPU")
        pts = {t: classical_peaks(np.asarray(movie[t]), scale, k_mad=k_mad) for t in range(n_frames)}
    return points_to_nodes(pts)


def merge_detections(primary, secondary, scale, min_dist_um=3.0):
    """Keep every `primary` node; add `secondary` nodes that are farther than min_dist_um from all primary nodes of that frame."""
    vs = np.asarray(scale, dtype=float)
    parts = [primary[["t", "z", "y", "x"]]]
    sec_by_t = {int(t): g for t, g in secondary.groupby("t")}
    pri_by_t = {int(t): g for t, g in primary.groupby("t")}
    for t, g in sec_by_t.items():
        if t not in pri_by_t:
            parts.append(g[["t", "z", "y", "x"]])
            continue
        d, _ = cKDTree(pri_by_t[t][["z", "y", "x"]].to_numpy() * vs).query(g[["z", "y", "x"]].to_numpy() * vs)
        parts.append(g.loc[d > min_dist_um, ["t", "z", "y", "x"]])
    out = pd.concat(parts, ignore_index=True).sort_values("t", kind="stable").reset_index(drop=True)
    out.insert(0, "node_id", np.arange(1, len(out) + 1))
    return out.astype("int64")


## 5. Linking with motion compensation, divisions and gap closing

In [6]:
def match_frames(pa, pb, max_um, dense_limit=4_000_000):
    """One-to-one matching of points pa (Na,3) to pb (Nb,3) in micrometres, total distance minimal, gate max_um.
    Splits the problem into independent clusters first, so it stays fast with thousands of cells per frame."""
    Na, Nb = len(pa), len(pb)
    if Na == 0 or Nb == 0:
        return []
    coo = cKDTree(pb).sparse_distance_matrix(cKDTree(pa), max_um, output_type="coo_matrix")
    if coo.nnz == 0:
        return []
    rb, ca, d = coo.row.astype(np.int64), coo.col.astype(np.int64), coo.data + 1e-9
    graph = csr_matrix((np.ones(len(d)), (ca, Na + rb)), shape=(Na + Nb, Na + Nb))
    _, lab = connected_components(graph, directed=False)
    comp = lab[ca]
    order = np.argsort(comp, kind="stable")
    cuts = np.flatnonzero(np.diff(comp[order])) + 1
    pairs = []
    for idx in np.split(order, cuts):
        a_u, a_inv = np.unique(ca[idx], return_inverse=True)
        b_u, b_inv = np.unique(rb[idx], return_inverse=True)
        if len(a_u) * len(b_u) <= dense_limit:
            big = 1e9
            cost = np.full((len(a_u), len(b_u)), big)
            cost[a_inv, b_inv] = d[idx]
            r, c = linear_sum_assignment(cost)
            pairs += [(int(a_u[i]), int(b_u[j])) for i, j in zip(r, c) if cost[i, j] < big]
        else:                                                        # huge cluster: greedy
            used_a, used_b = set(), set()
            for k in idx[np.argsort(d[idx])]:
                if ca[k] not in used_a and rb[k] not in used_b:
                    used_a.add(ca[k]); used_b.add(rb[k]); pairs.append((int(ca[k]), int(rb[k])))
    return pairs


import warnings


def estimate_flow(pa, pb, gate_um=14.0, k=12, radius_um=40.0, iters=2, tight_um=7.0):
    """Local motion field from a loose first matching: for each point of pa, the median displacement of the
    nearest matched points (non-rigid motion, e.g. tissue flow, plus the global drift)."""
    flow = np.zeros_like(pa)
    gate = gate_um
    for _ in range(iters):
        pairs = match_frames(pa + flow, pb, gate)
        if len(pairs) < 8:
            break
        ia, ib = (np.array(v) for v in zip(*pairs))
        disp, anchor = pb[ib] - pa[ia], pa[ia]
        kk = min(k, len(ia))
        d, nn = cKDTree(anchor).query(pa, k=kk, distance_upper_bound=radius_um)
        d, nn = d.reshape(len(pa), kk), nn.reshape(len(pa), kk)
        neigh = np.concatenate([disp, np.full((1, 3), np.nan)])[nn]            # missing neighbours -> index len(ia) -> NaN
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            local = np.nanmedian(neigh, axis=1)
        new = np.tile(np.median(disp, axis=0), (len(pa), 1))                   # fallback: global drift
        ok = ~np.isnan(local).any(axis=1)
        new[ok] = local[ok]
        flow, gate = new, tight_um
    return flow


def link_tracks(nodes_df, scale, max_link_um=None, flow=None,
                divisions=True, div_parent_um=4.6, div_sister_um=6.9, div_frame_frac=0.008, div_global_frac=0.0036,
                gap_close=True, gap_um=5.9, gap_max_frac=0.045):
    """Frame-to-frame linking: motion-compensated Hungarian matching, then safe divisions, then 1-frame gap closing.
    Returns (nodes_df incl. synthetic gap nodes, edges_df). Every edge goes from frame t to t+1."""
    max_link_um = MAX_LINK_UM if max_link_um is None else max_link_um
    flow = USE_FLOW if flow is None else flow
    vs = np.asarray(scale, dtype=float)
    nodes = nodes_df[["node_id", "t", "z", "y", "x"]].astype("int64").sort_values(["t", "node_id"]).reset_index(drop=True)
    N = len(nodes)
    by_t = {int(t): (g["node_id"].to_numpy(), g[["z", "y", "x"]].to_numpy() * vs) for t, g in nodes.groupby("t")}
    edges, div_cands, flows = [], [], {}
    for t in sorted(by_t):
        if t + 1 not in by_t:
            continue
        ida, pa = by_t[t]
        idb, pb = by_t[t + 1]
        fl = estimate_flow(pa, pb) if flow else np.zeros_like(pa)
        flows[t] = fl
        pred = pa + fl
        pairs = match_frames(pred, pb, max_link_um)
        child_of = {}
        for i, j in pairs:
            edges.append((int(ida[i]), int(idb[j])))
            child_of[i] = j
        if divisions and child_of:
            matched_b = set(child_of.values())
            free_b = [j for j in range(len(idb)) if j not in matched_b]
            if free_b:
                a_idx = np.array(sorted(child_of))
                d, k = cKDTree(pred[a_idx]).query(pb[free_b], distance_upper_bound=div_parent_um)
                best = {}
                for q, (dd, kk) in enumerate(zip(d, k)):
                    if not np.isfinite(dd):
                        continue
                    i, j = int(a_idx[kk]), free_b[q]
                    if np.linalg.norm(pb[j] - pb[child_of[i]]) <= div_sister_um and (i not in best or dd < best[i][0]):
                        best[i] = (dd, j)
                frame_cap = max(1, int(div_frame_frac * len(ida)))
                for dd, i, j in sorted((v[0], i, v[1]) for i, v in best.items())[:frame_cap]:
                    div_cands.append((dd, int(ida[i]), int(idb[j])))
    if divisions and div_cands:
        for dd, u, v in sorted(div_cands)[:max(1, int(div_global_frac * N))]:
            edges.append((u, v))

    extra_rows = []
    if gap_close and len(by_t) > 2:
        out_deg, in_deg = Counter(u for u, _ in edges), Counter(v for _, v in edges)
        next_id, cap, added = int(nodes["node_id"].max()) + 1, int(gap_max_frac * N), 0
        for t in sorted(by_t):
            if t + 2 not in by_t or added >= cap:
                continue
            ida, pa = by_t[t]
            idc, pc = by_t[t + 2]
            A = [i for i, n in enumerate(ida) if out_deg[int(n)] == 0]
            C = [j for j, n in enumerate(idc) if in_deg[int(n)] == 0]
            if not A or not C:
                continue
            pa_pred = pa + (2 * flows[t] if t in flows else 0)
            kk = min(3, len(C))
            d, k = cKDTree(pc[C]).query(pa_pred[A], k=kk, distance_upper_bound=gap_um)
            d, k = d.reshape(len(A), kk), k.reshape(len(A), kk)
            cands = sorted((d[q, w], A[q], C[k[q, w]]) for q in range(len(A)) for w in range(kk) if np.isfinite(d[q, w]))
            used_a, used_c = set(), set()
            for dd, i, j in cands:
                if i in used_a or j in used_c or added >= cap:
                    continue
                used_a.add(i); used_c.add(j)
                mid = np.rint((pa[i] + pc[j]) / 2 / vs).astype(np.int64)
                extra_rows.append((next_id, t + 1, int(mid[0]), int(mid[1]), int(mid[2])))
                edges += [(int(ida[i]), next_id), (next_id, int(idc[j]))]
                next_id += 1; added += 1
    nodes_out = pd.concat([nodes, pd.DataFrame(extra_rows, columns=nodes.columns, dtype="int64")], ignore_index=True)
    edges_df = pd.DataFrame(edges, columns=["source_id", "target_id"], dtype="int64").drop_duplicates().reset_index(drop=True)
    return nodes_out, edges_df


## 6. Local metric (official edge-Jaccard rules) and self-test

In [7]:
def local_edge_metrics(pred_nodes, pred_edges, gt_nodes, gt_edges, scale, tol_um=7.0):
    """Local version of the competition's edge Jaccard (rules from metrics.md): optimal 7 um node matching per frame,
    TP = predicted edge between matched nodes that is a GT edge, FP = predicted edge that contradicts a GT edge,
    FN = GT edge not recovered. Unmatched predicted nodes are NOT penalised (node-count factor not included).
    Pass pred_nodes / gt_nodes already restricted to the frames you want to score."""
    vs = np.asarray(scale, dtype=float)
    zyx = ["z", "y", "x"]
    gt_ids = set(gt_nodes.node_id.astype(int))
    gt_e = gt_edges[gt_edges.source_id.isin(gt_ids) & gt_edges.target_id.isin(gt_ids)]
    gt_in, gt_out = defaultdict(set), defaultdict(set)
    for u, v in zip(gt_e.source_id.astype(int), gt_e.target_id.astype(int)):
        gt_out[u].add(v); gt_in[v].add(u)
    gt_edge_set = {(u, v) for u, vs_ in gt_out.items() for v in vs_}
    p2g = {}
    for t, g in gt_nodes.groupby("t"):
        p = pred_nodes[pred_nodes.t == t]
        if p.empty:
            continue
        cost = cdist(p[zyx].to_numpy() * vs, g[zyx].to_numpy() * vs)
        cost = np.where(cost <= tol_um, cost, 1e6)
        r, c = linear_sum_assignment(cost)
        pid, gid = p["node_id"].to_numpy(), g["node_id"].to_numpy()
        for i, j in zip(r, c):
            if cost[i, j] < 1e6:
                p2g[int(pid[i])] = int(gid[j])
    keep = set(pred_nodes.node_id.astype(int))
    pe = pred_edges[pred_edges.source_id.isin(keep) & pred_edges.target_id.isin(keep)]
    tp = fp = 0
    hit = set()
    for u, v in zip(pe.source_id.astype(int), pe.target_id.astype(int)):
        gu, gv = p2g.get(u), p2g.get(v)
        if gu is not None and gv is not None and (gu, gv) in gt_edge_set:
            tp += 1; hit.add((gu, gv))
            continue
        if (gv is not None and any(s != gu for s in gt_in.get(gv, ()))) or \
           (gu is not None and any(tg != gv for tg in gt_out.get(gu, ()))):
            fp += 1
    return {"TP": tp, "FP": fp, "FN": len(gt_edge_set) - len(hit), "n_gt_nodes": len(gt_nodes), "n_matched": len(p2g),
            "n_pred_nodes": len(pred_nodes), "n_frames": int(pred_nodes.t.nunique())}


def pool(results):
    """Micro-average several metric dicts (the competition pools TP/FP/FN over all samples before computing the Jaccard)."""
    s = {k: sum(r[k] for r in results) for k in results[0]} if results else {}
    if not s:
        return {}
    return {"edge_jaccard": round(s["TP"] / max(s["TP"] + s["FP"] + s["FN"], 1), 4), "TP": s["TP"], "FP": s["FP"], "FN": s["FN"],
            "node_recall": round(s["n_matched"] / max(s["n_gt_nodes"], 1), 3), "nodes/frame": round(s["n_pred_nodes"] / max(s["n_frames"], 1), 1)}


In [8]:
def run_selftest():
    """No data needed. (1) perfect detections must score ~1 (2) with a spatially varying motion field plus noise, the
    flow-compensated linker must clearly beat plain nearest matching (3) a division is found."""
    rng = np.random.default_rng(0)
    scale = (1.625, 0.40625, 0.40625)
    n, T = 500, 6
    pos0 = rng.uniform([2, 20, 20], [60, 240, 240], (n, 3)) * np.array(scale)          # micrometres, ~12 um spacing
    rows, gt_edges, nid, last = [], [], 1, {}
    for t in range(T):
        # drift 4 um/frame + shear that grows with y; 0.5 um noise
        shift = np.array([1.0, 4.0, 3.0]) * t + np.column_stack([np.zeros(n), np.zeros(n), 0.06 * pos0[:, 1] * t])
        p = pos0 + shift + rng.normal(0, 0.5, (n, 3))
        vox = np.rint(p / np.array(scale)).astype(int)
        for k in range(n):
            rows.append((nid, t, *vox[k]))
            if k in last:
                gt_edges.append((last[k], nid))
            last[k] = nid; nid += 1
    gt = pd.DataFrame(rows, columns=["node_id", "t", "z", "y", "x"])
    gte = pd.DataFrame(gt_edges, columns=["source_id", "target_id"])
    res = {}
    for flow in (False, True):
        n_out, e = link_tracks(gt, scale, max_link_um=10.0, flow=flow, divisions=False, gap_close=False)
        res[flow] = pool([local_edge_metrics(n_out, e, gt, gte, scale)])["edge_jaccard"]
    print(f"synthetic drifting tissue: edge Jaccard without flow {res[False]:.3f} -> with flow {res[True]:.3f}")
    assert res[True] > 0.9 and res[True] > res[False] + 0.1, res
    d = pd.DataFrame([(1, 0, 10, 100, 100), (2, 1, 10, 100, 104), (3, 1, 10, 100, 96)], columns=["node_id", "t", "z", "y", "x"])
    _, e2 = link_tracks(d, scale, flow=False, divisions=True, gap_close=False)
    assert sorted(map(tuple, e2.values.tolist())) == [(1, 2), (1, 3)], e2
    print("linking self-test OK")

run_selftest()


synthetic drifting tissue: edge Jaccard without flow 0.732 -> with flow 0.987
linking self-test OK


## 7. Load detector

In [9]:
model = None
if DETECTOR == "unet":
    cands = find_files("/kaggle/input", ["unet_detector*.pt"], skip_names=(COMPETITION,)) + [str(p) for p in WORKING_DIR.glob("unet_detector*.pt")]
    if cands:
        try:
            model = load_detector(cands[0])
            print("loaded detector:", cands[0], "| parameters:", round(sum(p.numel() for p in model.parameters()) / 1e6, 2), "M")
        except Exception:
            traceback.print_exc()
    if model is None:
        print("!! no usable unet_detector.pt found -> falling back to the blob detector (attach your unet-detector dataset)")
        DETECTOR = "blob"
print("DETECTOR:", DETECTOR)


def read_frame(movie, t):
    return np.asarray(movie[t])


def blob_points(frame, scale):
    """Blob detector on one frame (GPU if available and verified, else CPU)."""
    if device.type == "cuda" and GPU_BLOB:
        return classical_peaks_gpu(frame, scale, k_mad=K_MAD)
    return classical_peaks(frame, scale, k_mad=K_MAD)


def detect_points(frame, scale, low_thr=0.05):
    """(points, scores) of one frame. For the U-Net scores are probabilities >= low_thr (filter later by threshold)."""
    if DETECTOR == "unet":
        return detect_frame(model, frame, scale, thr=low_thr)
    pts = blob_points(frame, scale)
    return pts, np.ones(len(pts), dtype=np.float32)


loaded detector: /kaggle/input/notebooks/parth2157/biohub-a-train-detector-ipynb/unet_detector.pt | parameters: 8.09 M
DETECTOR: unet


## 8. Validation on held-out movies (windows spread over time)

Compare `blob` with `unet`, and flow off/on. **Gate:** keep the U-Net only if its `edge_jaccard` is clearly (about +0.05) above the blob row.

In [10]:
def collect_windows(stems, detector, starts, width, k_blob=None):
    """Run the detector once on every frame of the validation windows. Returns a list of window records."""
    recs = []
    for s in stems:
        arr, scale = open_movie(TRAIN_DIR, s)
        gn, ge = load_geff_graph(TRAIN_DIR / f"{s}.geff")
        T = arr.shape[0]
        for st in starts:
            ts = [t for t in range(st, st + width) if t < T]
            if len(ts) < 2:
                continue
            dets = {}
            for t in ts:
                fr = read_frame(arr, t)
                dets[t] = detector(fr, scale)
            recs.append({"stem": s, "ts": ts, "scale": scale, "gt_nodes": gn[gn.t.isin(ts)], "gt_edges": ge, "dets": dets})
    return recs


def score_windows(recs, thr, max_link_um, flow):
    results = []
    for r in recs:
        pts = {t: p[sc >= thr] for t, (p, sc) in r["dets"].items()}
        nodes = points_to_nodes(pts)
        if nodes.empty:
            results.append({"TP": 0, "FP": 0, "FN": 0, "n_gt_nodes": len(r["gt_nodes"]), "n_matched": 0, "n_pred_nodes": 0, "n_frames": len(r["ts"])})
            continue
        n_out, e = link_tracks(nodes, r["scale"], max_link_um=max_link_um, flow=flow)
        results.append(local_edge_metrics(n_out, e, r["gt_nodes"], r["gt_edges"], r["scale"]))
    return pool(results)


all_stems = list_stems(TRAIN_DIR)
fit_stems, hold_stems = split_stems(all_stems, N_HOLDOUT, SEED)
print(f"{len(hold_stems)} held-out movies (never used for training in notebook A)")
rows, recs_unet = [], None
if hold_stems:
    t0 = time.time()
    if DETECTOR == "unet":
        recs_unet = collect_windows(hold_stems, lambda fr, sc: detect_points(fr, sc), WINDOW_STARTS, WINDOW_WIDTH)
        print(f"U-Net detections on {len(recs_unet)} windows: {time.time() - t0:.0f}s")
    if COMPARE_BLOB or DETECTOR == "blob":
        if device.type == "cuda" and not GPU_CHECKED:
            a0, s0 = open_movie(TRAIN_DIR, hold_stems[0])
            check_gpu_blob(a0, s0)
        blob_fn = lambda fr, sc: (lambda p: (p, np.ones(len(p))))(blob_points(fr, sc))
        recs_blob = collect_windows(hold_stems[:6], blob_fn, WINDOW_STARTS, WINDOW_WIDTH)
        for flow in (False, True):
            rows.append({"detector": f"blob K_MAD={K_MAD}", "thr": "-", "link_um": 10.0, "flow": flow, **score_windows(recs_blob, 0.5, 10.0, flow)})
    if recs_unet:
        for flow in (False, True):
            rows.append({"detector": "unet", "thr": 0.2, "link_um": 10.0, "flow": flow, **score_windows(recs_unet, 0.2, 10.0, flow)})
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print("no training movies available for validation -> using the SETTINGS values")


20 held-out movies (never used for training in notebook A)
U-Net detections on 100 windows: 3036s
      detector  thr  link_um  flow  edge_jaccard   TP  FP  FN  node_recall  nodes/frame
blob K_MAD=4.0    -     10.0 False        0.5364  177  33 120        0.677        243.7
blob K_MAD=4.0    -     10.0  True        0.5621  181  25 116        0.677        243.8
          unet  0.2     10.0 False        0.7729 1882 331 222        0.979        261.8
          unet  0.2     10.0  True        0.7849 1894 309 210        0.978        261.8


## 9. Tune threshold / link distance / flow

In [11]:
if AUTO_TUNE and DETECTOR == "unet" and recs_unet:
    grid = []
    for thr in THR_GRID:
        for lk in LINK_GRID:
            for flow in (True, False):
                grid.append({"thr": thr, "link_um": lk, "flow": flow, **score_windows(recs_unet, thr, lk, flow)})
    gdf = pd.DataFrame(grid).sort_values("edge_jaccard", ascending=False).reset_index(drop=True)
    print(gdf.head(10).to_string(index=False))
    near = gdf[gdf.edge_jaccard >= gdf.edge_jaccard.iloc[0] - 0.005].sort_values(["nodes/frame", "thr"], ascending=[True, False])   # ties -> fewer nodes, higher threshold
    best = near.iloc[0]
    THRESHOLD, MAX_LINK_UM, USE_FLOW = float(best.thr), float(best.link_um), bool(best.flow)
    print(f"\n-> using THRESHOLD={THRESHOLD}, MAX_LINK_UM={MAX_LINK_UM}, USE_FLOW={USE_FLOW}  (held-out edge Jaccard {best.edge_jaccard}, "
          f"node recall {best.node_recall}, {best['nodes/frame']} nodes/frame)")
else:
    print("keeping SETTINGS: THRESHOLD =", THRESHOLD, "| MAX_LINK_UM =", MAX_LINK_UM, "| USE_FLOW =", USE_FLOW)


 thr  link_um  flow  edge_jaccard   TP  FP  FN  node_recall  nodes/frame
0.15      8.0  True        0.8987 1978  97 126        0.979        268.2
0.10      8.0  True        0.8972 1981 104 123        0.979        272.3
0.20      8.0  True        0.8941 1975 105 129        0.978        262.9
0.30      8.0  True        0.8887 1964 106 140        0.973        249.1
0.10      8.0 False        0.8871 1973 120 131        0.980        272.3
0.15      8.0 False        0.8867 1972 120 132        0.980        268.1
0.20      8.0 False        0.8837 1968 123 136        0.979        262.8
0.30      8.0 False        0.8820 1958 116 146        0.973        249.0
0.40      8.0  True        0.8710 1925 106 179        0.954        226.9
0.40      8.0 False        0.8638 1916 114 188        0.954        226.7

-> using THRESHOLD=0.2, MAX_LINK_UM=8.0, USE_FLOW=True  (held-out edge Jaccard 0.8941, node recall 0.978, 262.9 nodes/frame)


## 10. Predict the test movies and write `submission.csv`

In [12]:
def build_submission_rows(dataset, nodes_df, edges_df):
    cols = ["id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
    n = pd.DataFrame({"dataset": dataset, "row_type": "node", "node_id": nodes_df["node_id"].to_numpy(),
                      "t": nodes_df["t"].to_numpy(), "z": nodes_df["z"].to_numpy(),
                      "y": nodes_df["y"].to_numpy(), "x": nodes_df["x"].to_numpy()})
    e = pd.DataFrame({"dataset": dataset, "row_type": "edge",
                      "source_id": edges_df["source_id"].to_numpy(), "target_id": edges_df["target_id"].to_numpy()})
    out = pd.concat([n, e], ignore_index=True)
    for c in ["node_id", "t", "z", "y", "x", "source_id", "target_id"]:
        out[c] = out[c].fillna(-1).astype("int64")                  # sample_submission uses -1, not blanks
    out.insert(0, "id", 0)
    return out.reindex(columns=cols)


def write_submission(parts, path):
    sub = pd.concat(parts, ignore_index=True)
    sub["id"] = np.arange(len(sub))
    sub.to_csv(path, index=False)
    return sub




In [13]:
def predict_movie(stem):
    movie, scale = open_movie(TEST_DIR, stem)
    n_frames = movie.shape[0]

    def run(mode_detector):
        pts = {}
        with ThreadPoolExecutor(max_workers=3) as pool_:
            for t, fr in enumerate(pool_.map(lambda i: read_frame(movie, i), range(n_frames))):
                p, sc = mode_detector(fr, scale)
                pts[t] = p[sc >= THRESHOLD] if DETECTOR == "unet" else p
        return link_tracks(points_to_nodes(pts), scale)

    try:
        return run(detect_points)
    except Exception:
        traceback.print_exc()
        print(f"   {stem}: failed -> redoing with the blob detector")
        return link_tracks(classical_detect_movie(movie, scale, verbose=False), scale)


test_stems = list_stems(TEST_DIR)
assert test_stems, f"No .zarr movies found in {TEST_DIR}"
parts, t_all = [], time.time()
for k, s in enumerate(test_stems, 1):
    t0 = time.time()
    nodes_out, edges = predict_movie(s)
    n_fr = max(int(nodes_out.t.max()) + 1, 1)
    forks = int((edges.source_id.value_counts() > 1).sum())
    done_min = (time.time() - t_all) / 60
    print(f"[{k}/{len(test_stems)}] {s}: {len(nodes_out):,} nodes ({len(nodes_out) / n_fr:.0f}/frame), {len(edges):,} edges, "
          f"{forks} divisions, {time.time() - t0:.0f}s | elapsed {done_min:.0f} min, ETA {done_min / k * (len(test_stems) - k):.0f} min")
    parts.append(build_submission_rows(s, nodes_out, edges))

SUBMISSION_PATH = WORKING_DIR / "submission.csv"
sub = write_submission(parts, SUBMISSION_PATH)
print("wrote", SUBMISSION_PATH, "| rows:", f"{len(sub):,}", "| nodes:", f"{(sub.row_type == 'node').sum():,}", "| edges:", f"{(sub.row_type == 'edge').sum():,}")
if SAMPLE_SUBMISSION.exists():
    sample = pd.read_csv(SAMPLE_SUBMISSION)
    print("columns match sample :", list(sample.columns) == list(sub.columns))
    print("dtypes match sample  :", bool((sample.dtypes.astype(str) == sub.dtypes.astype(str)).all()))
    print("datasets match sample:", set(sample.dataset) == set(sub.dataset))
sub.head()


[1/4] 44b6_0113de3b: 25,481 nodes (255/frame), 24,473 edges, 14 divisions, 786s | elapsed 13 min, ETA 39 min
[2/4] 44b6_0b24845f: 47,960 nodes (480/frame), 43,238 edges, 117 divisions, 840s | elapsed 27 min, ETA 27 min
[3/4] 6bba_05b6850b: 7,114 nodes (71/frame), 6,602 edges, 5 divisions, 837s | elapsed 41 min, ETA 14 min
[4/4] 6bba_05db0fb1: 69,133 nodes (691/frame), 66,122 edges, 72 divisions, 868s | elapsed 56 min, ETA 0 min
wrote /kaggle/working/submission.csv | rows: 290,123 | nodes: 149,688 | edges: 140,435
columns match sample : True
dtypes match sample  : True
datasets match sample: True


,id,dataset,row_type,node_id,t,z,y,x,source_id,target_id
0,0,44b6_0113de3b,node,1,0,1,6,52,-1,-1
1,1,44b6_0113de3b,node,2,0,1,8,70,-1,-1
2,2,44b6_0113de3b,node,3,0,1,98,34,-1,-1
3,3,44b6_0113de3b,node,4,0,1,106,74,-1,-1
4,4,44b6_0113de3b,node,5,0,1,126,70,-1,-1
